# Predicción de tiempos y costos de atención mediante regresión lineal

Este notebook guía una práctica aplicada a un **equipo de atención de servicios**. El objetivo es usar datos históricos para responder dos preguntas de negocio:

1. **¿Cuánto tiempo podría tomar atender una solicitud?**
2. **¿Cuál podría ser su costo de atención?**

 El código principal ya está estructurado para que el trabajo se concentre en **ejecutar, modificar parámetros definidos e interpretar resultados**.

**Objetivos del notebook**
- Explorar un conjunto de solicitudes de servicio;
- Construir una regresión lineal simple;
- Observar la función de costo y el descenso del gradiente;
- Ampliar el modelo a múltiples características;
- Aplicar escalamiento e ingeniería de características;
- Generar predicciones para casos nuevos.


## 0. Preparación del entorno

**Instalación de dependencias**

Ejecuta la siguiente celda una sola vez. Instala las librerías incluidas en `requirements.txt` dentro del mismo entorno utilizado por el kernel del notebook.

In [ ]:
print("Instalando dependencias del laboratorio...")
%pip install -r requirements.txt --progress-bar on
print("Dependencias instaladas correctamente.")

### Cargar librerías y datos

En esta celda se preparan las herramientas que utilizaremos durante toda la práctica:

- **NumPy:** operaciones numéricas y vectorización.
- **pandas:** lectura y análisis de datos tabulares.
- **Matplotlib:** visualización de resultados.
- **scikit-learn:** entrenamiento, escalamiento y evaluación de modelos.

También se localiza y carga `solicitudes_servicio.csv`. Si esta celda termina correctamente y muestra **600 registros**, el entorno está listo.


In [ ]:
from pathlib import Path

# Librerías para cálculo numérico y análisis de datos.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Herramientas de scikit-learn utilizadas a lo largo del laboratorio.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Opciones para visualizar tablas con mayor claridad.
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# Buscamos el CSV en ubicaciones comunes para facilitar la ejecución desde VS Code.
candidatas = [
    Path("solicitudes_servicio.csv"),
    Path("materiales/solicitudes_servicio.csv"),
    Path("Capitulo01/materiales/solicitudes_servicio.csv"),
]
RUTA_DATOS = next((ruta for ruta in candidatas if ruta.exists()), None)

if RUTA_DATOS is None:
    raise FileNotFoundError(
        "No se encontró solicitudes_servicio.csv. "
        "Verifique que el notebook y el CSV estén dentro de la carpeta materiales."
    )

# Cada fila del archivo representa una solicitud de servicio histórica.
datos = pd.read_csv(RUTA_DATOS)

print(f"Archivo cargado: {RUTA_DATOS.resolve()}")
print(f"Registros: {len(datos):,} | Columnas: {datos.shape[1]}")

## Reto 1. Comprender el problema y explorar los datos

### Contexto del caso de uso

Un equipo de operaciones recibe solicitudes con diferentes niveles de impacto y complejidad. Antes de asignar recursos, le interesa estimar **tiempo** y **costo** utilizando información conocida al inicio de la atención.

En este reto comprobarás qué información contiene el histórico y qué relaciones parecen existir antes de entrenar cualquier modelo.

### Qué debes observar

- Qué representa cada fila.
- Qué variables podrían ayudar a explicar el tiempo o el costo.
- Si existen datos faltantes.
- Si `incidencias_reportadas` presenta una relación visible con `tiempo_atencion_horas`.


In [ ]:
# Mostramos las primeras solicitudes para entender cómo está estructurado el dataset.
display(datos.head(8))

# Estas son las variables que utilizaremos durante el laboratorio.
columnas_modelado = [
    "usuarios_afectados",
    "incidencias_reportadas",
    "complejidad_servicio",
    "horas_estimadas",
    "cambios_recientes",
    "canales_involucrados",
    "tiempo_atencion_horas",
    "costo_atencion_usd",
]

# describe() permite revisar rango, promedio, dispersión y posibles valores atípicos.
display(datos[columnas_modelado].describe().T)

# Un modelo requiere conocer la calidad de sus datos antes de entrenarse.
print("Valores faltantes:")
display(datos[columnas_modelado].isna().sum().to_frame("faltantes"))

### Visualizar una relación inicial

Vamos a representar `incidencias_reportadas` frente a `tiempo_atencion_horas`. Si los puntos muestran una tendencia ascendente, tendremos una primera evidencia de que esa variable puede aportar información al modelo.

La gráfica **no demuestra causalidad**. Solo permite observar una asociación en los datos históricos.

In [ ]:
plt.figure(figsize=(8, 5))

# Cada punto representa una solicitud histórica.
plt.scatter(
    datos["incidencias_reportadas"],
    datos["tiempo_atencion_horas"],
    alpha=0.45,
)

plt.title("Incidencias reportadas vs. tiempo de atención")
plt.xlabel("Incidencias reportadas")
plt.ylabel("Tiempo de atención (horas)")
plt.show()


### Reflexión aplicada

Responde antes de continuar:

- ¿Se aprecia una tendencia creciente?
- ¿Una sola variable parece suficiente para explicar todo el tiempo de atención?
- En un escenario real, ¿qué otras variables podrían modificar el tiempo aunque el número de incidencias sea igual?

-------------------------------------

## Reto 2. Construir una regresión lineal simple

### Contexto del caso de uso

Queremos crear una primera estimación sencilla del **tiempo de atención** usando únicamente `incidencias_reportadas`.

Este modelo sirve como **línea base**: nos permite entender qué puede explicar una sola característica antes de incorporar información adicional.

### Qué hace el código

1. Separa la característica `X` de la variable objetivo `y`.
2. Reserva 20 % de los datos para prueba.
3. Entrena una regresión lineal con el 80 % restante.
4. Predice sobre casos que el modelo no utilizó durante el entrenamiento.
5. Calcula métricas para cuantificar el error.


In [ ]:
# X contiene la información que el modelo conoce antes de predecir.
X_simple = datos[["incidencias_reportadas"]]

# y contiene el resultado histórico que queremos aprender a estimar.
y_tiempo = datos["tiempo_atencion_horas"]

# Separamos entrenamiento y prueba para evaluar el modelo en datos no vistos.
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_simple,
    y_tiempo,
    test_size=0.20,
    random_state=42,
)

# Creamos y entrenamos el modelo lineal.
modelo_simple = LinearRegression()
modelo_simple.fit(X_train_s, y_train_s)

# Generamos predicciones sobre el conjunto de prueba.
pred_simple = modelo_simple.predict(X_test_s)

# MAE y RMSE expresan error en horas. R² indica cuánto de la variabilidad explica el modelo.
mae_simple = mean_absolute_error(y_test_s, pred_simple)
rmse_simple = np.sqrt(mean_squared_error(y_test_s, pred_simple))
r2_simple = r2_score(y_test_s, pred_simple)

print(f"Intercepto: {modelo_simple.intercept_:.3f} horas")
print(f"Pendiente: {modelo_simple.coef_[0]:.3f} horas por incidencia")
print(f"MAE:  {mae_simple:.3f} horas")
print(f"RMSE: {rmse_simple:.3f} horas")
print(f"R²:   {r2_simple:.3f}")


### Interpretar la recta aprendida

La pendiente indica cuánto cambia el tiempo estimado cuando aumenta en una unidad el número de incidencias, manteniendo el modelo en su forma más simple.

La siguiente gráfica superpone la recta aprendida sobre los casos históricos de entrenamiento.

In [ ]:
# Creamos valores ordenados para dibujar una línea continua.
x_linea = np.linspace(
    datos["incidencias_reportadas"].min(),
    datos["incidencias_reportadas"].max(),
    100,
).reshape(-1, 1)

# El modelo calcula el tiempo estimado para cada punto de la línea.
y_linea = modelo_simple.predict(x_linea)

plt.figure(figsize=(8, 5))
plt.scatter(
    X_train_s["incidencias_reportadas"],
    y_train_s,
    alpha=0.35,
    label="Entrenamiento",
)
plt.plot(x_linea, y_linea, linewidth=2, label="Modelo lineal")
plt.title("Modelo lineal simple")
plt.xlabel("Incidencias reportadas")
plt.ylabel("Tiempo de atención (horas)")
plt.legend()
plt.show()


### Reflexión aplicada

- ¿Qué significa la pendiente obtenida para un responsable de operaciones?
- ¿El MAE sería aceptable para planificar tiempos de atención?
- ¿Qué riesgo existiría si se utilizara este modelo como única fuente para comprometer un tiempo con un cliente?

------------------

## Reto 3. Observar la función de costo y el descenso del gradiente

### Contexto del caso de uso

Cuando entrenamos un modelo, necesitamos encontrar los parámetros que produzcan el menor error posible. El **descenso del gradiente** modifica progresivamente esos parámetros y la **función de costo** permite medir si cada actualización mejora o empeora el modelo.

Aquí no programarás el algoritmo desde cero. El código está preparado para que experimentes con la **tasa de aprendizaje (`alpha`)** y observes su efecto.

### Relación con un caso real

Una tasa demasiado pequeña hace que el aprendizaje sea lento. Una tasa adecuada converge de forma estable. Una tasa excesiva puede hacer que el entrenamiento se vuelva inestable y el error crezca.


In [ ]:
def costo_lineal(x, y, w, b):
    """Calcula el error cuadrático medio dividido entre 2."""
    pred = w * x + b
    return np.mean((pred - y) ** 2) / 2


def descenso_gradiente(x, y, alpha=0.1, iteraciones=80):
    """Ajusta pendiente (w) e intercepto (b) de forma iterativa."""

    # Comenzamos sin conocimiento previo: w = 0 y b = 0.
    w = 0.0
    b = 0.0
    historial = []
    m = len(y)

    for _ in range(iteraciones):
        # 1. Calculamos la predicción actual.
        pred = w * x + b

        # 2. Medimos cuánto se aleja de los valores reales.
        error = pred - y

        # 3. Guardamos el costo para observar la convergencia.
        historial.append(np.mean(error ** 2) / 2)

        # 4. Calculamos los gradientes de forma vectorizada.
        dw = np.dot(error, x) / m
        db = np.mean(error)

        # 5. Actualizamos los parámetros en dirección contraria al gradiente.
        w -= alpha * dw
        b -= alpha * db

    return w, b, historial


# Convertimos las columnas de pandas a arreglos NumPy.
x_gd = X_train_s["incidencias_reportadas"].to_numpy(dtype=float)
y_gd = y_train_s.to_numpy(dtype=float)

# Escalamos manualmente la característica para que el experimento con alpha sea estable y comparable.
x_media, x_std = x_gd.mean(), x_gd.std()
x_gd_escalado = (x_gd - x_media) / x_std

# MODIFICA estos valores para experimentar con distintas velocidades de aprendizaje.
ALPHAS = [0.01, 0.10, 2.10]
resultados_alpha = {}

plt.figure(figsize=(9, 5))

for alpha in ALPHAS:
    w, b, hist = descenso_gradiente(
        x_gd_escalado,
        y_gd,
        alpha=alpha,
        iteraciones=80,
    )

    resultados_alpha[alpha] = {
        "w": w,
        "b": b,
        "costo_inicial": hist[0],
        "costo_final": hist[-1],
    }

    plt.plot(hist, label=f"alpha={alpha}")

plt.yscale("log")
plt.title("Convergencia del descenso del gradiente")
plt.xlabel("Iteración")
plt.ylabel("Costo J(w,b) - escala log")
plt.legend()
plt.show()

display(pd.DataFrame(resultados_alpha).T)


### Reflexión aplicada

Compara las tres curvas:

- ¿Cuál aprende demasiado lento?
- ¿Cuál converge de forma rápida y estable?
- ¿Cuál vuelve inestable el entrenamiento?

En un proyecto real, `alpha` es un **hiperparámetro**: no lo aprende el modelo, sino que se define y valida durante el desarrollo.

-------

## Reto 4. Estimar costo con múltiples características

### Contexto del caso de uso

El costo de atención difícilmente depende de una sola variable. Una solicitud con muchos usuarios afectados, alta complejidad, múltiples incidencias y varias horas estimadas puede requerir más recursos.

Por eso construiremos un modelo multivariable para estimar `costo_atencion_usd`.

### Qué aprenderás aquí

- Por qué varias características pueden explicar mejor un resultado.
- Para qué sirve `StandardScaler`.
- Cómo un `Pipeline` mantiene juntos preparación y modelo.
- Cómo interpretar MAE, RMSE y R² en un caso de costos.

In [ ]:
# Variables disponibles antes de finalizar la atención y que podrían explicar el costo.
CARACTERISTICAS_BASE = [
    "usuarios_afectados",
    "incidencias_reportadas",
    "complejidad_servicio",
    "horas_estimadas",
    "cambios_recientes",
    "canales_involucrados",
]

# Usamos una única partición para que las comparaciones posteriores sean justas.
indices_train, indices_test = train_test_split(
    datos.index,
    test_size=0.20,
    random_state=42,
)

X_train = datos.loc[indices_train, CARACTERISTICAS_BASE]
X_test = datos.loc[indices_test, CARACTERISTICAS_BASE]
y_train = datos.loc[indices_train, "costo_atencion_usd"]
y_test = datos.loc[indices_test, "costo_atencion_usd"]

# Pipeline: primero estandariza las características y después entrena la regresión.
modelo_base = Pipeline([
    ("escalador", StandardScaler()),
    ("regresion", LinearRegression()),
])

modelo_base.fit(X_train, y_train)
pred_base = modelo_base.predict(X_test)


def metricas(y_real, y_pred):
    """Calcula métricas comparables para todos los modelos de costo."""
    return {
        "MAE": mean_absolute_error(y_real, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_real, y_pred)),
        "R2": r2_score(y_real, y_pred),
    }


metricas_base = metricas(y_test, pred_base)
display(pd.DataFrame([metricas_base], index=["Modelo multivariable base"]))

# Como las variables fueron estandarizadas, la magnitud de los coeficientes ayuda a comparar su peso relativo.
coef_base = pd.Series(
    modelo_base.named_steps["regresion"].coef_,
    index=CARACTERISTICAS_BASE,
    name="coeficiente_estandarizado",
).sort_values(key=np.abs, ascending=False)

display(coef_base.to_frame())


### Vectorización: calcular varias contribuciones en una sola operación

La regresión multivariable combina cada característica con su coeficiente. En vez de calcular cada término por separado, NumPy puede realizarlo mediante multiplicación matricial.

Esta misma idea permite que los modelos procesen muchas observaciones y características de forma eficiente.

In [ ]:
# Tomamos tres solicitudes de prueba y las escalamos igual que durante el entrenamiento.
X_demo = modelo_base.named_steps["escalador"].transform(X_test.iloc[:3])

# Recuperamos los coeficientes aprendidos por la regresión.
w_demo = modelo_base.named_steps["regresion"].coef_
b_demo = modelo_base.named_steps["regresion"].intercept_

# Operación vectorizada: X @ w + b calcula las tres predicciones en una sola expresión.
pred_vectorizada = X_demo @ w_demo + b_demo

# El pipeline debe producir exactamente el mismo resultado.
pred_pipeline = modelo_base.predict(X_test.iloc[:3])

print("Predicción vectorizada:", np.round(pred_vectorizada, 2))
print("Predicción del pipeline:", np.round(pred_pipeline, 2))
print("¿Coinciden?", np.allclose(pred_vectorizada, pred_pipeline))

### Reflexión aplicada

- ¿El modelo de costo explica más variabilidad que el modelo simple de tiempo?
- ¿Qué características parecen tener mayor relación con el costo estimado?
- ¿Por qué sería riesgoso interpretar un coeficiente como prueba de causalidad?

---

## Reto 5. Mejorar el modelo mediante ingeniería de características

### Contexto del caso de uso

A veces la información útil no está representada directamente en una sola columna. Una incidencia puede tener un efecto distinto dependiendo de la **complejidad del servicio**.

Crearemos la característica:

`carga_operativa = incidencias_reportadas × complejidad_servicio`

Esta variable intenta representar la interacción entre volumen y dificultad.

Después compararemos tres alternativas:

1. Modelo base.
2. Modelo base + `carga_operativa`.
3. Modelo con características polinómicas de grado 2.

La meta no es elegir el modelo más complejo, sino comprobar cuál generaliza mejor sobre datos de prueba.

In [ ]:
# Copiamos el dataset para conservar las columnas originales.
datos_ext = datos.copy()

# Ingeniería de características: combinamos volumen de incidencias y complejidad.
datos_ext["carga_operativa"] = (
    datos_ext["incidencias_reportadas"]
    * datos_ext["complejidad_servicio"]
)

CARACTERISTICAS_EXT = CARACTERISTICAS_BASE + ["carga_operativa"]

# Modelo con la nueva característica diseñada manualmente.
modelo_ext = Pipeline([
    ("escalador", StandardScaler()),
    ("regresion", LinearRegression()),
])

modelo_ext.fit(
    datos_ext.loc[indices_train, CARACTERISTICAS_EXT],
    y_train,
)

pred_ext = modelo_ext.predict(
    datos_ext.loc[indices_test, CARACTERISTICAS_EXT]
)
metricas_ext = metricas(y_test, pred_ext)

# Variante polinómica: genera automáticamente términos cuadrados e interacciones de grado 2.
modelo_poly = Pipeline([
    ("polinomios", PolynomialFeatures(degree=2, include_bias=False)),
    ("escalador", StandardScaler()),
    ("regresion", LinearRegression()),
])

modelo_poly.fit(
    datos.loc[indices_train, CARACTERISTICAS_BASE],
    y_train,
)

pred_poly = modelo_poly.predict(
    datos.loc[indices_test, CARACTERISTICAS_BASE]
)
metricas_poly = metricas(y_test, pred_poly)

# Comparamos todos los modelos sobre exactamente el mismo conjunto de prueba.
comparacion = pd.DataFrame([
    {"modelo": "Base", **metricas_base},
    {"modelo": "+ carga_operativa", **metricas_ext},
    {"modelo": "Polinómico grado 2", **metricas_poly},
]).set_index("modelo")

display(comparacion)

### Analizar residuos

Un residuo es la diferencia entre el valor real y el valor predicho. Revisarlos ayuda a detectar situaciones donde el modelo falla de forma sistemática.

- Residuo positivo: el modelo **subestimó** el costo.
- Residuo negativo: el modelo **sobreestimó** el costo.
- Patrón alrededor de cero: comportamiento más consistente con un ajuste adecuado.


In [ ]:
# Calculamos la diferencia entre costo real y costo estimado.
residuos = y_test.to_numpy() - pred_ext

plt.figure(figsize=(8, 5))
plt.scatter(pred_ext, residuos, alpha=0.55)

# La línea cero representa una predicción sin error.
plt.axhline(0, linestyle="--")

plt.title("Residuos del modelo con ingeniería de características")
plt.xlabel("Costo predicho (USD)")
plt.ylabel("Residuo: real - predicho (USD)")
plt.show()


### Reflexión aplicada

Compara **MAE, RMSE y R²** de los tres modelos y responde:

- ¿La característica `carga_operativa` aporta una mejora observable?
- ¿La expansión polinómica mejora realmente el resultado de prueba?
- ¿Qué modelo elegirías si además de precisión necesitas explicar la lógica a un responsable de negocio?

> Un modelo más complejo no se considera automáticamente mejor. La decisión debe equilibrar desempeño, generalización e interpretabilidad.


---

## Reto 6. Predecir nuevas solicitudes y convertir el modelo en apoyo para una decisión

### Contexto del caso de uso

El valor de un modelo aparece cuando puede aplicarse a solicitudes que todavía no han sido atendidas. Simularemos tres casos con diferentes niveles de carga y utilizaremos el modelo seleccionado para estimar su costo.

El resultado puede apoyar actividades como:

- planificación de recursos;
- priorización de revisión;
- estimación preliminar de esfuerzo;
- identificación de casos que requieren validación adicional.

La predicción es una **estimación**, no una autorización automática ni un costo definitivo.

In [ ]:
# Tres solicitudes nuevas: baja, media y alta carga operativa.
casos_nuevos = pd.DataFrame([
    {
        "usuarios_afectados": 35,
        "incidencias_reportadas": 2,
        "complejidad_servicio": 1,
        "horas_estimadas": 1.8,
        "cambios_recientes": 0,
        "canales_involucrados": 1,
    },
    {
        "usuarios_afectados": 180,
        "incidencias_reportadas": 7,
        "complejidad_servicio": 3,
        "horas_estimadas": 4.2,
        "cambios_recientes": 1,
        "canales_involucrados": 3,
    },
    {
        "usuarios_afectados": 420,
        "incidencias_reportadas": 14,
        "complejidad_servicio": 5,
        "horas_estimadas": 7.5,
        "cambios_recientes": 1,
        "canales_involucrados": 4,
    },
])

# La nueva característica debe calcularse exactamente igual que durante el entrenamiento.
casos_nuevos["carga_operativa"] = (
    casos_nuevos["incidencias_reportadas"]
    * casos_nuevos["complejidad_servicio"]
)

# Generamos la estimación de costo para cada solicitud nueva.
casos_nuevos["costo_estimado_usd"] = modelo_ext.predict(
    casos_nuevos[CARACTERISTICAS_EXT]
)

display(casos_nuevos)


### Revisar qué variables pesan más en el modelo

Los coeficientes estandarizados permiten comparar la magnitud relativa de las características dentro de este modelo. Deben interpretarse como asociaciones aprendidas de los datos, no como relaciones causales.


In [ ]:
import json

# Ordenamos los coeficientes por magnitud absoluta para facilitar su lectura.
coef_ext = pd.Series(
    modelo_ext.named_steps["regresion"].coef_,
    index=CARACTERISTICAS_EXT,
    name="coeficiente_estandarizado",
).sort_values(key=np.abs, ascending=False)

display(coef_ext.to_frame())

# Consolidamos las métricas para revisar el recorrido completo del laboratorio.
resumen_metricas = {
    "simple_tiempo": {
        "MAE": mae_simple,
        "RMSE": rmse_simple,
        "R2": r2_simple,
    },
    "costo_base": metricas_base,
    "costo_extendido": metricas_ext,
    "costo_polinomico": metricas_poly,
}

print(
    json.dumps(
        {
            k: {m: round(float(v), 4) for m, v in d.items()}
            for k, d in resumen_metricas.items()
        },
        indent=2,
    )
)


---

## Cierre del laboratorio

Conecta los resultados técnicos con el caso de uso y responde:

1. ¿Qué diferencia observaste entre utilizar una sola característica y utilizar varias?
2. ¿Qué ocurrió cuando modificaste la tasa de aprendizaje?
3. ¿Qué aportó la característica `carga_operativa`?
4. ¿Qué modelo elegirías para estimar costo y qué evidencia respalda tu decisión?
5. ¿Qué revisión humana mantendrías antes de utilizar una predicción para planificación operativa?

### Resultado del aprendizaje

Al finalizar no solo has ejecutado una regresión. Has recorrido un flujo completo de aprendizaje supervisado: **datos → entrenamiento → función de costo → optimización → evaluación → mejora de características → predicción → interpretación para una decisión**.